<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_05_4_generators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 6: Convolutional Neural Networks (CNN) for Computer Vision**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Module 5 Material

- Part 5.1: Image Processing in Python [[Video]](https://www.youtube.com/watch?v=Sob7VDb4xh8&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Notebook]](t81_558_class_05_1_python_images.ipynb)
- Part 5.2: Using Convolutional Neural Networks [[Video]](https://www.youtube.com/watch?v=jL0_lOpEwSk&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Notebook]](t81_558_class_05_2_cnn.ipynb)
- Part 5.3: Using Pretrained Neural Networks[[Video]](https://www.youtube.com/watch?v=W2T-dfiHYSo&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Notebook]](t81_558_class_05_3_vision_transfer.ipynb)
- **Part 5.4: Looking at Generators and Image Augmentation** [[Video]](https://www.youtube.com/watch?v=20JoEmQb810&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Notebook]](t81_558_class_05_4_generators.ipynb)
- Part 5.5: Recognizing Multiple Images with YOLOv5 [[Video]](https://www.youtube.com/watch?v=7Uu1n9Tp0Mk&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Notebook]](t81_558_class_05_5_yolo.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running the correct version of TensorFlow.

In [ ]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 3.2)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Part 5.4: Inside Augmentation

The [ImageDataGenerator](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image/ImageDataGenerator) class provides many options for image augmentation.  Deciding which augmentations to use can impact the effectiveness of your model. This part will visualize some of these augmentations that you might use to train your neural network. We begin by loading a sample image to augment.

In [ ]:
import urllib.request
import shutil
from IPython.display import Image

URL =  "https://data.heatonresearch.com/images/wustl/photos/landscape.jpg"

if COLAB:
  LOCAL_IMG_FILE = "/content/landscape.jpg"
else:
  # Modify for your system
  LOCAL_IMG_FILE = "/Users/jeff/temp/landscape.jpg"

with urllib.request.urlopen(URL) as response, \
  open(LOCAL_IMG_FILE, 'wb') as out_file:
    shutil.copyfileobj(response, out_file)

Image(filename=LOCAL_IMG_FILE)

Next, we introduce a simple utility function to visualize four images sampled from any generator.

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.utils import make_grid
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt


def visualize_generator(img_file, transform):
    # Load the requested image
    img = Image.open(img_file).resize((256,256))

    # Apply transformations to the image
    imgs = [transform(img) for _ in range(4)]
    imgs = torch.stack(imgs)

    # Display the augmented images in a grid format
    grid = make_grid(imgs, nrow=2)

    # Convert to numpy for plotting
    grid_np = grid.numpy().transpose(1, 2, 0)
    grid_np = (grid_np * 255).astype(np.uint8)

    plt.figure(figsize=(15, 15))
    plt.axis("off")
    plt.imshow(grid_np)
    plt.show()

We begin by flipping the image. Some images may not make sense to flip, such as this landscape.  However, if you expect "noise" in your data where some images may be flipped, then this augmentation may be useful, even if it violates physical reality.

In [ ]:
# Define the transformations using PyTorch's torchvision.transforms
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor()
])

visualize_generator(LOCAL_IMG_FILE, transform)

Next, we will try moving the image. Notice how part of the image is missing? There are various ways to fill in the missing data, as controlled by **padding_mode**. In this case, we simply use the nearest pixel to fill. It is also possible to rotate images.

In [ ]:
transform = transforms.Compose([
    transforms.RandomAffine(degrees=0, translate=(0.5, 0)), # assuming the image width is 400 for a shift of [-200,200]
    transforms.Pad(padding=200, fill=0, padding_mode='edge'),  # pad with nearest value
    transforms.ToTensor()
])

visualize_generator(LOCAL_IMG_FILE, transform)

We can also adjust brightness.

In [ ]:
transform = transforms.Compose([
    transforms.ColorJitter(brightness=(0, 1)), # Adjust brightness between [0, 1]
    transforms.ToTensor()
])

visualize_generator(LOCAL_IMG_FILE, transform)

Shearing may not be appropriate for all image types, it stretches the image.

In [ ]:
transform = transforms.Compose([
    transforms.RandomAffine(degrees=0, shear=30),  # Apply shear transformation
    transforms.ToTensor()
])

visualize_generator(LOCAL_IMG_FILE, transform)

It is also possible to rotate images.

In [ ]:
# Define the transformations using PyTorch's torchvision.transforms
transform = transforms.Compose([
    transforms.RandomAffine(degrees=30),  # Rotate the image up to 30 degrees
    transforms.ToTensor()
])

visualize_generator(LOCAL_IMG_FILE, transform)